# Biokataliza przemysłowa — 100 kg glukozy

**Jesteś technologiem.** Masz 150 kg skrobi. Wyprodukuj **co najmniej 100 kg glukozy w maksymalnie 4 godziny** i wybierz wariant o najniższym koszcie.

**Co robisz?** Zmieniasz tylko temperaturę i dawkę enzymu, uruchamiasz symulację, porównujesz 9 wariantów i odpowiadasz na pytania. Nie musisz pisać funkcji ani rozwiązywać równań.

Model i koszty są **hipotetyczne i dydaktyczne**. Przyjmujemy skrobię wstępnie upłynnioną; pomijamy operacje przygotowania i oczyszczania.

## 1. Uruchom gotowy model

Maksymalna teoretyczna masa glukozy: `150 × 180/162 = 166,7 kg` (w hydrolizie przyłącza się woda).

Koszt = **85 zł × kg enzymu** + **1,1 zł × (T − 25)** + **czas [h] × [50 + 0,07 × (T − 25)²] zł/h**.

W modelu enzym przyspiesza reakcję, ale traci aktywność z czasem — tym szybciej, im wyższa temperatura.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

P_MAX = 150 * 180 / 162
CEL = 100

def symulacja(T, enzym):
    k = 65 * np.exp(0.045 * (T - 50))
    kd = 0.12 * np.exp(0.085 * (T - 50))
    def rownanie(t, y):
        reszta = max(P_MAX - y[0], 0)
        return [k * enzym * np.exp(-kd*t) * reszta / (35 + reszta)]
    czas = np.linspace(0, 4, 1001)
    wyn = solve_ivp(rownanie, (0, 4), [0], t_eval=czas,
                    rtol=1e-8, atol=1e-10)
    return wyn.t, wyn.y[0]

def ocen(T, enzym):
    t, P = symulacja(T, enzym)
    if P[-1] < CEL:
        return None, None
    h = float(np.interp(CEL, P, t))
    koszt = 85*enzym + 1.1*(T-25) + h*(50+0.07*(T-25)**2)
    return h, koszt

print('Maksymalnie:', round(P_MAX, 1), 'kg glukozy')
print('Wymagana konwersja:', round(100*CEL/P_MAX), '%')

## 2. Własna próba — zmień DWIE liczby

Sprawdź na przykład 40°C i 1 kg enzymu, a następnie 50°C i 2 kg. Odczytaj wykres i koszt.

In [ ]:
# ZMIENIAJ TYLKO PONIŻSZE WARTOŚCI
T = 50         # temperatura [°C]
enzym = 2.0    # dawka [kg]

czas, glukoza = symulacja(T, enzym)
h, koszt = ocen(T, enzym)
plt.plot(czas, glukoza)
plt.axhline(100, color='black', linestyle='--', label='Cel 100 kg')
plt.xlabel('Czas [h]')
plt.ylabel('Glukoza [kg]')
plt.title(f'{T}°C, {enzym} kg enzymu')
plt.grid(alpha=0.3)
plt.legend()
plt.show()
if h is None:
    print('Nie osiągnięto 100 kg w 4 h')
else:
    print(f'100 kg po {h:.2f} h; koszt {koszt:.2f} zł')

## 3. Porównaj dziewięć wariantów

**Gotowy kod — tylko uruchom.** Sprawdź, które warianty spełniają cel i który jest najtańszy.

In [ ]:
wyniki = []
print('Temp. | Enzym | Czas do 100 kg | Koszt')
print('-'*45)
for T in [40, 50, 60]:
    for E in [1, 2, 3]:
        h, koszt = ocen(T, E)
        wyniki.append((T, E, h, koszt))
        if h is None:
            print(f'{T:>5}°C | {E} kg  | NIE OSIĄGNIĘTO CELU')
        else:
            print(f'{T:>5}°C | {E} kg  | {h:>6.2f} h       | {koszt:>7.2f} zł')

poprawne = [w for w in wyniki if w[2] is not None]
najlepszy = min(poprawne, key=lambda w: w[3])
print('Najtańszy wariant z tabeli:')
print(f'{najlepszy[0]}°C, {najlepszy[1]} kg enzymu, '
      f'{najlepszy[2]:.2f} h, koszt {najlepszy[3]:.2f} zł')

## 4. Krótkie sprawozdanie (5 pytań)

1. Jaka konwersja skrobi (w przeliczeniu na glukozę) jest potrzebna, aby uzyskać 100 kg glukozy?
2. Które warianty z tabeli **nie osiągają celu** w ciągu 4 h?
3. Jaka jest **najtańsza** kombinacja temperatury i dawki enzymu z tabeli? Jaki daje koszt i czas?
4. Dlaczego wyższa temperatura **nie zawsze** oznacza niższy koszt?
5. Sprawdź dodatkowy wariant **45°C, 1,5 kg enzymu**. Czy jest tańszy od najlepszego z dziewięciu?